# Semana 12 — Vector Databases: Qdrant, pgvector e Metadata Filtering
**Fase 3: Advanced RAG, Search e Reranking** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/03-Phase-3-Advanced-RAG/notebooks/semana_12_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-12-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/03-Phase-3-Advanced-RAG/docs/Semana-12-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_12_gabarito.ipynb`.

In [ ]:
!pip -q install qdrant-client sentence-transformers

In [ ]:
import numpy as np
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# --- Corpus de demonstração: base de conhecimento fictícia de uma empresa ---
import random
def corpus_demo(seed=7):
    """Devolve (docs, consultas).
    docs: lista de {id, texto, area, ano}.
    consultas: lista de {q, resposta, relevantes: {doc_id: grau}}; grau 2 = contém a resposta, 1 = mesmo assunto."""
    rnd = random.Random(seed)
    fatos = {
        "ferias": [("O colaborador tem direito a 30 dias de férias após 12 meses de trabalho.", "Quantos dias de descanso anual eu tenho?"),
                   ("As férias podem ser divididas em até três períodos, um deles com no mínimo 14 dias.", "Posso fracionar meu período de descanso?"),
                   ("O pedido de férias deve ser feito no portal RH com 45 dias de antecedência.", "Com que antecedência solicito minha folga anual?")],
        "reembolso": [("Despesas de viagem são reembolsadas em até 10 dias úteis após o envio dos recibos.", "Em quanto tempo recebo de volta o que gastei viajando?"),
                      ("O limite diário de alimentação em viagens nacionais é de R$ 120.", "Qual o teto de gasto com refeição em deslocamento a trabalho?"),
                      ("Corridas de aplicativo só são reembolsadas entre 22h e 6h ou em deslocamentos a clientes.", "A empresa paga meu Uber?")],
        "seguranca": [("Senhas corporativas expiram a cada 90 dias e exigem autenticação em dois fatores.", "De quanto em quanto tempo preciso trocar minha credencial?"),
                      ("Dados de clientes classificados como confidenciais não podem ser enviados a ferramentas externas de IA.", "Posso colar informações de cliente em um chatbot público?"),
                      ("Incidentes de segurança devem ser reportados ao SOC em até 2 horas.", "Qual o prazo para avisar sobre um vazamento?")],
        "ti": [("Notebooks são substituídos a cada 36 meses mediante chamado no service desk.", "Quando posso trocar meu computador?"),
               ("O acesso à VPN exige o cliente GlobalConnect e certificado pessoal.", "Como me conecto à rede da empresa de casa?"),
               ("Licenças de software adicionais precisam de aprovação do gestor e do time de compras.", "Como consigo um programa novo?")],
        "beneficios": [("O plano de saúde cobre dependentes legais sem custo adicional até 21 anos.", "Meu filho pode entrar no convênio médico?"),
                       ("O vale-refeição é creditado no último dia útil de cada mês.", "Em que dia cai o cartão de alimentação?"),
                       ("A empresa subsidia 50% de cursos de idiomas, limitado a R$ 400 por mês.", "Existe ajuda para estudar inglês?")],
    }
    codigos = [("O código de erro VPN-4031 indica certificado pessoal expirado; renove pelo portal de identidades.", "ti", "O que significa o erro VPN-4031?"),
               ("O formulário RH-207 é usado para solicitar licença não remunerada.", "ferias", "Para que serve o formulário RH-207?"),
               ("O centro de custo CC-8812 deve ser usado para despesas de treinamento externo.", "reembolso", "Qual centro de custo uso: CC-8812 é para quê?"),
               ("A política SEG-014 define a classificação de dados em público, interno e confidencial.", "seguranca", "O que diz a política SEG-014?")]
    projetos = ["Atlas", "Boreal", "Cometa", "Delta", "Éden"]
    armadilhas = ["O time do projeto {p} volta de férias em janeiro e retoma as entregas do trimestre.",
                  "No projeto {p}, o reembolso ao cliente foi aprovado após a revisão do contrato.",
                  "A equipe do projeto {p} trocou a senha do ambiente de testes durante a migração.",
                  "O projeto {p} comprou notebooks novos para o laboratório de dados.",
                  "O plano de saúde financeira do projeto {p} prevê corte de 10% nos custos de nuvem."]
    ruido = ["Este procedimento foi revisado pelo comitê interno.", "Dúvidas devem ser encaminhadas ao gestor imediato.",
             "A política vale para todas as unidades do grupo.", "Exceções exigem aprovação formal da diretoria."]
    docs, consultas = [], []
    def novo(texto, area):
        docs.append({"id": len(docs), "texto": texto, "area": area, "ano": rnd.choice([2023, 2024, 2025])}); return docs[-1]["id"]
    por_area = {}
    for area, itens in fatos.items():
        for fato, pergunta in itens:
            i = novo(f"{fato} {rnd.choice(ruido)}", area); por_area.setdefault(area, []).append(i)
            consultas.append({"q": pergunta, "resposta": fato, "relevantes": {i: 2}, "area": area})
    for texto, area, pergunta in codigos:
        i = novo(texto, area); por_area[area].append(i)
        consultas.append({"q": pergunta, "resposta": texto, "relevantes": {i: 2}, "area": area})
    for c in consultas:
        for j in por_area[c["area"]]:
            c["relevantes"].setdefault(j, 1)
    for p in projetos:
        for a in armadilhas:
            novo(a.format(p=p), "projetos")
    return docs, consultas

def manual_demo(docs):
    """Junta os documentos de política em um manual longo em Markdown (para exercícios de chunking)."""
    titulos = {"ferias": "Férias e licenças", "reembolso": "Viagens e reembolsos", "seguranca": "Segurança da informação",
               "ti": "Equipamentos e acesso", "beneficios": "Benefícios"}
    partes = ["# Manual do Colaborador\n\nEste manual reúne as políticas internas vigentes."]
    for area, titulo in titulos.items():
        frases = [d["texto"] for d in docs if d["area"] == area]
        partes.append(f"## {titulo}\n\n" + "\n\n".join(" ".join(frases[i:i + 2]) for i in range(0, len(frases), 2)))
    return "\n\n".join(partes)

docs, consultas = corpus_demo()
modelo = SentenceTransformer(MODELO_EMB)
E = modelo.encode([d["texto"] for d in docs], normalize_embeddings=True, convert_to_numpy=True)
emb = lambda q: modelo.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0]
print(E.shape)

## Passo 1 — Criar a collection
Qdrant em modo local (em memória): mesma API do servidor, sem infraestrutura. Crie a collection `politicas` com cosine distance e a dimensão dos seus embeddings.

In [ ]:
cli = QdrantClient(":memory:")
# [SEU CÓDIGO AQUI] cli.create_collection(...) com VectorParams(size, distance) e hnsw_config

## Passo 2 — Ingestão com payload
Cada point leva `id`, `vector` e `payload` (texto, área, ano).

In [ ]:
# [SEU CÓDIGO AQUI] monte a lista `pontos` (PointStruct) e faça o upsert
print("pontos na coleção:", cli.count("politicas", exact=True).count)

## Passo 3 — Search

In [ ]:
def buscar(q, k=3, filtro=None):
    # [SEU CÓDIGO AQUI] cli.query_points(...) com query, limit, query_filter e with_payload
    return r.points

def mostrar(pts):
    for p in pts: print(f"   {p.score:.3f} [{p.payload['area']:10s} {p.payload['ano']}] {p.payload['texto'][:70]}")

q = "O que preciso fazer sobre a troca de senha?"
mostrar(buscar(q, 5))

## Passo 4 — Metadata filtering
Restrinja a busca à área `seguranca`. Depois, combine: área `seguranca` **e** ano ≥ 2024.

In [ ]:
# [SEU CÓDIGO AQUI] defina f_area e f_area_ano
print("só segurança:"); mostrar(buscar(q, 5, f_area))
print("segurança e ano >= 2024:"); mostrar(buscar(q, 5, f_area_ano))

## Passo 5 — Post-filtering perde resultados
Suponha que este usuário **não pode ver** documentos da área `projetos`. Simule o post-filtering: busque o top-5 sem filtro e só depois descarte os documentos de `projetos`. Compare quantos resultados sobram com a busca que aplica o filtro durante a query.

In [ ]:
permitido = models.Filter(must_not=[models.FieldCondition(key="area", match=models.MatchValue(value="projetos"))])
q2 = "O que aconteceu com as férias e o reembolso do projeto?"
# [SEU CÓDIGO AQUI] pos = top-5 sem filtro, filtrado depois; integrado = busca com o filtro `permitido`
print(f"pós-filtro: {len(pos)} resultados | filtro integrado: {len(integrado)} resultados")
mostrar(integrado)

Se `projetos` fosse uma área confidencial, o que o post-filtering arriscaria?

## Passo 6 — Ciclo de vida: atualizar, marcar e excluir
1. A política de senhas mudou: faça *upsert* do mesmo `id` com texto e vetor novos.
2. Marque um point como não vigente com `set_payload`.
3. Exclua um point.

In [ ]:
alvo = next(d for d in docs if "Senhas corporativas" in d["texto"])
novo_texto = "Senhas corporativas expiram a cada 180 dias e exigem chave física de segurança."
# [SEU CÓDIGO AQUI] upsert do point atualizado, set_payload em docs[1] e delete de docs[2]
print("pontos após exclusão:", cli.count("politicas", exact=True).count)
print(cli.retrieve("politicas", ids=[alvo["id"], docs[1]["id"]], with_payload=True))

## Passo 7 — Payload index
Em collections grandes, campos usados em filtros devem ser indexados.

In [ ]:
# [SEU CÓDIGO AQUI] crie um payload index para o campo "area"
print(cli.get_collection("politicas").points_count, "pontos | índice de payload criado")

## Passo 8 (opcional) — pgvector no Colab
As células abaixo instalam PostgreSQL e compilam o `pgvector` dentro do Colab. Levam alguns minutos e podem exigir ajuste de versão; pule se o tempo estiver curto. O SQL equivalente está na leitura de conceito.

In [ ]:
RODAR_PGVECTOR = False   # mude para True para executar
if RODAR_PGVECTOR:
    import subprocess
    for cmd in ["apt-get -qq update", "apt-get -qq install -y postgresql postgresql-server-dev-all build-essential git",
                "rm -rf pgvector && git clone -q https://github.com/pgvector/pgvector.git && cd pgvector && make -s && make -s install",
                "service postgresql start",
                "sudo -u postgres psql -c \"ALTER USER postgres PASSWORD 'postgres'\" -c 'CREATE EXTENSION IF NOT EXISTS vector'",
                "pip -q install psycopg2-binary"]:
        subprocess.run(cmd, shell=True, check=True)
    import psycopg2
    con = psycopg2.connect(host="localhost", user="postgres", password="postgres", dbname="postgres"); con.autocommit = True
    cur = con.cursor()
    cur.execute("DROP TABLE IF EXISTS documentos")
    cur.execute(f"CREATE TABLE documentos (id int PRIMARY KEY, texto text, area text, ano int, embedding vector({E.shape[1]}))")
    for i, d in enumerate(docs):
        cur.execute("INSERT INTO documentos VALUES (%s, %s, %s, %s, %s)", (d["id"], d["texto"], d["area"], d["ano"], str(E[i].tolist())))
    cur.execute("CREATE INDEX ON documentos USING hnsw (embedding vector_cosine_ops)")
    cur.execute("""SELECT texto, 1 - (embedding <=> %s::vector) AS sim FROM documentos
                   WHERE area = 'seguranca' ORDER BY embedding <=> %s::vector LIMIT 3""", (str(emb(q).tolist()),) * 2)
    for linha in cur.fetchall(): print(round(linha[1], 3), linha[0][:80])

## Teste Prático de Entrevista — Pre-Filtering Multi-Tenant em Vector DB
Um entrevistador pede: *"Em um banco vetorial multi-tenant, post-filtering pode devolver menos de `k` itens ou vazar informação. Implemente `pre_filtered_search(records, query_vec, tenant_id, top_k)` filtrando `tenant_id` ANTES de calcular a cosine similarity e retornando os `top_k` `(id, score)` em ordem decrescente."* 

In [ ]:
import math


def pre_filtered_search(records, query_vec, tenant_id, top_k):
    """Filtra por tenant_id antes de rankear por cosine similarity."""
    # [SEU CÓDIGO AQUI]
    pass


recs = [
    {"id": "a1", "tenant": "acme", "vec": [1.0, 0.0]},
    {"id": "b1", "tenant": "beta", "vec": [0.9, 0.1]},
    {"id": "b2", "tenant": "beta", "vec": [0.8, 0.2]},
    {"id": "a2", "tenant": "acme", "vec": [0.5, 0.5]},
]
res = pre_filtered_search(recs, [1.0, 0.0], tenant_id="acme", top_k=2)
assert [r[0] for r in res] == ["a1", "a2"], f"Esperado a1 e a2, veio {res}"
print("OK! Pre-filtered search:", res)